In [ ]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.9/208.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.8/761.8 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.1/128.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.4/399.4 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.8 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00


In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)



In [ ]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
from datetime import date, timedelta

today = date.today()
seven_days = today + timedelta(days=7)

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 38.2527,
    "longitude": -85.7585,
    "hourly": [
        "temperature_2m", "relative_humidity_2m", "precipitation_probability",
        "precipitation", "wind_speed_10m", "soil_temperature_0cm",
        "soil_moisture_0_to_1cm"
    ],
    "timezone": "auto",
    "wind_speed_unit": "mph",
    "temperature_unit": "fahrenheit",
    "precipitation_unit": "inch",
    "start_date": today.isoformat(),
    "end_date": seven_days.isoformat(),
}

responses = openmeteo.weather_api(url, params = params)



In [ ]:
# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")



Coordinates: 38.26070022583008°N -85.758544921875°E
Elevation: 139.0 m asl
Timezone: b'America/Kentucky/Louisville'b'GMT-4'
Timezone difference to GMT+0: -14400s


In [ ]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()
hourly_soil_temperature_0cm = hourly.Variables(5).ValuesAsNumpy()
hourly_soil_moisture_0_to_1cm = hourly.Variables(6).ValuesAsNumpy()

dates = pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
    freq=pd.Timedelta(seconds=hourly.Interval()),
    inclusive="left"
)



In [ ]:
# Convert to local time, then strip timezone for PostgreSQL
dates = dates.tz_convert(response.Timezone().decode()).tz_localize(None)

hourly_data = {"date": dates}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["soil_temperature_0cm"] = hourly_soil_temperature_0cm
hourly_data["soil_moisture_0_to_1cm"] = hourly_soil_moisture_0_to_1cm

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

hourly_dataframe["date"] = pd.to_datetime(hourly_dataframe["date"])

hourly_dataframe.to_csv("weather_soil.csv", index=False)



Hourly data
                    date  temperature_2m  relative_humidity_2m  \
0   2026-05-23 00:00:00       68.066597                  93.0   
1   2026-05-23 01:00:00       68.066597                  96.0   
2   2026-05-23 02:00:00       67.796600                  96.0   
3   2026-05-23 03:00:00       68.966599                  95.0   
4   2026-05-23 04:00:00       68.876602                  93.0   
..                  ...             ...                   ...   
187 2026-05-30 19:00:00       61.456100                  61.0   
188 2026-05-30 20:00:00       58.936104                  66.0   
189 2026-05-30 21:00:00       56.686100                  68.0   
190 2026-05-30 22:00:00       54.436100                  70.0   
191 2026-05-30 23:00:00       52.636101                  71.0   

     precipitation_probability  precipitation  wind_speed_10m  \
0                         82.0       0.141732        5.188683   
1                         71.0       0.125984        4.529580   
2         

In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
from datetime import date, timedelta

# -----------------------------------------------------------
# Setup Open-Meteo client
# -----------------------------------------------------------
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# -----------------------------------------------------------
# Dynamic date handling with API limit protection
# -----------------------------------------------------------
today = date.today()
seven_days = today + timedelta(days=7)

# Air Quality API maximum allowed date
api_max_date = date(2026, 5, 29)

# Ensure end_date does not exceed API limit
end_date = min(seven_days, api_max_date)

# -----------------------------------------------------------
# Air Quality + Pollen API request
# -----------------------------------------------------------
url = "https://air-quality-api.open-meteo.com/v1/air-quality"

params = {
    "latitude": 38.2527,
    "longitude": -85.7585,
    "hourly": [
        "pm10", "pm2_5", "carbon_monoxide", "ozone", "nitrogen_dioxide",
        "sulphur_dioxide", "us_aqi", "us_aqi_pm2_5", "us_aqi_pm10",
        "us_aqi_nitrogen_dioxide", "us_aqi_carbon_monoxide",
        "us_aqi_ozone", "us_aqi_sulphur_dioxide",
        "grass_pollen", "ragweed_pollen", "olive_pollen",
        "mugwort_pollen", "birch_pollen", "alder_pollen"
    ],
    "timezone": "auto",
    "domains": "cams_global",
    "start_date": today.isoformat(),
    "end_date": end_date.isoformat(),
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]

# -----------------------------------------------------------
# Extract hourly variables
# -----------------------------------------------------------
hourly = response.Hourly()

hourly_pm10 = hourly.Variables(0).ValuesAsNumpy()
hourly_pm2_5 = hourly.Variables(1).ValuesAsNumpy()
hourly_carbon_monoxide = hourly.Variables(2).ValuesAsNumpy()
hourly_ozone = hourly.Variables(3).ValuesAsNumpy()
hourly_nitrogen_dioxide = hourly.Variables(4).ValuesAsNumpy()
hourly_sulphur_dioxide = hourly.Variables(5).ValuesAsNumpy()
hourly_us_aqi = hourly.Variables(6).ValuesAsNumpy()
hourly_us_aqi_pm2_5 = hourly.Variables(7).ValuesAsNumpy()
hourly_us_aqi_pm10 = hourly.Variables(8).ValuesAsNumpy()
hourly_us_aqi_nitrogen_dioxide = hourly.Variables(9).ValuesAsNumpy()
hourly_us_aqi_carbon_monoxide = hourly.Variables(10).ValuesAsNumpy()
hourly_us_aqi_ozone = hourly.Variables(11).ValuesAsNumpy()
hourly_us_aqi_sulphur_dioxide = hourly.Variables(12).ValuesAsNumpy()
hourly_grass_pollen = hourly.Variables(13).ValuesAsNumpy()
hourly_ragweed_pollen = hourly.Variables(14).ValuesAsNumpy()
hourly_olive_pollen = hourly.Variables(15).ValuesAsNumpy()
hourly_mugwort_pollen = hourly.Variables(16).ValuesAsNumpy()
hourly_birch_pollen = hourly.Variables(17).ValuesAsNumpy()
hourly_alder_pollen = hourly.Variables(18).ValuesAsNumpy()

# -----------------------------------------------------------
# Build PostgreSQL-safe date column
# -----------------------------------------------------------
dates = pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
    freq=pd.Timedelta(seconds=hourly.Interval()),
    inclusive="left"
)

# Convert to local timezone, then strip timezone for PostgreSQL
dates = dates.tz_convert(response.Timezone().decode()).tz_localize(None)

hourly_data = {"date": dates}

# -----------------------------------------------------------
# Add variables to dictionary
# -----------------------------------------------------------
hourly_data["pm10"] = hourly_pm10
hourly_data["pm2_5"] = hourly_pm2_5
hourly_data["carbon_monoxide"] = hourly_carbon_monoxide
hourly_data["ozone"] = hourly_ozone
hourly_data["nitrogen_dioxide"] = hourly_nitrogen_dioxide
hourly_data["sulphur_dioxide"] = hourly_sulphur_dioxide
hourly_data["us_aqi"] = hourly_us_aqi
hourly_data["us_aqi_pm2_5"] = hourly_us_aqi_pm2_5
hourly_data["us_aqi_pm10"] = hourly_us_aqi_pm10
hourly_data["us_aqi_nitrogen_dioxide"] = hourly_us_aqi_nitrogen_dioxide
hourly_data["us_aqi_carbon_monoxide"] = hourly_us_aqi_carbon_monoxide
hourly_data["us_aqi_ozone"] = hourly_us_aqi_ozone
hourly_data["us_aqi_sulphur_dioxide"] = hourly_us_aqi_sulphur_dioxide
hourly_data["grass_pollen"] = hourly_grass_pollen
hourly_data["ragweed_pollen"] = hourly_ragweed_pollen
hourly_data["olive_pollen"] = hourly_olive_pollen
hourly_data["mugwort_pollen"] = hourly_mugwort_pollen
hourly_data["birch_pollen"] = hourly_birch_pollen
hourly_data["alder_pollen"] = hourly_alder_pollen

# -----------------------------------------------------------
# Create DataFrame + save CSV
# -----------------------------------------------------------
hourly_dataframe = pd.DataFrame(hourly_data)
hourly_dataframe.to_csv("air_quality_pollen.csv", index=False)


In [ ]:
import pandas as pd

# Load CSVs
weather = pd.read_csv("weather_soil.csv")
air_pollen = pd.read_csv("air_quality_pollen.csv")

# Ensure date columns are parsed as datetime and timezone-free
weather["date"] = pd.to_datetime(weather["date"]).dt.tz_localize(None)
air_pollen["date"] = pd.to_datetime(air_pollen["date"]).dt.tz_localize(None)

# Merge on clean datetime column
merged = weather.merge(air_pollen, on="date", how="inner")

# Sort by date for consistency
merged = merged.sort_values("date")

# Save final merged dataset
merged.to_csv("merged_open_meteo.csv", index=False)


In [ ]:
merged.head()

,date,temperature_2m,relative_humidity_2m,precipitation_probability,precipitation,wind_speed_10m,soil_temperature_0cm,soil_moisture_0_to_1cm,pm10,pm2_5,...,us_aqi_nitrogen_dioxide,us_aqi_carbon_monoxide,us_aqi_ozone,us_aqi_sulphur_dioxide,grass_pollen,ragweed_pollen,olive_pollen,mugwort_pollen,birch_pollen,alder_pollen
0,2026-05-23 00:00:00,68.0666,93.0,82.0,0.141732,5.188683,66.837204,0.370,2.0,1.9,...,1.920804,1.736715,17.857141,0.109051,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-05-23 01:00:00,68.0666,96.0,71.0,0.125984,4.529580,66.477196,0.364,1.6,1.6,...,1.822301,1.763285,18.031076,0.163577,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-05-23 02:00:00,67.7966,96.0,42.0,0.000000,3.325540,66.747200,0.360,1.6,1.5,...,1.773050,1.766908,18.262987,0.218103,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-05-23 03:00:00,68.9666,95.0,32.0,0.000000,3.257121,66.837204,0.361,1.6,1.6,...,1.625296,1.739130,18.842764,0.218103,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-05-23 04:00:00,68.8766,93.0,36.0,0.015748,5.592500,67.287200,0.358,1.8,1.8,...,1.526793,1.692029,19.886364,0.272628,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
merged.columns = (
    merged.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("__+", "_", regex=True)
)


In [ ]:
merged["planting_readiness"] = (
    (merged["soil_temperature_0cm"].clip(50, 80) - 50) / 30 * 40 +
    (1 - merged["precipitation_probability"] / 100) * 20 +
    (1 - merged["wind_speed_10m"] / 20).clip(0, 1) * 20 +
    (1 - merged["soil_moisture_0_to_1cm"].clip(0, 0.5) / 0.5) * 20
).clip(0, 100)


In [ ]:
merged["allergy_risk"] = (
    (merged["pm2_5"] / 35).clip(0, 1) * 25 +
    (merged["ozone"] / 70).clip(0, 1) * 15 +
    (merged[["grass_pollen", "ragweed_pollen", "birch_pollen",
             "alder_pollen", "mugwort_pollen", "olive_pollen"]]
     .fillna(0)
     .mean(axis=1) / 200).clip(0, 1) * 60
).clip(0, 100)


In [ ]:
# 1. High Wind Flag
merged["high_wind_flag"] = (merged["wind_speed_10m"] > 15).astype(int)

# 2. Rain Expected Flag
merged["rain_expected_flag"] = (merged["precipitation_probability"] > 50).astype(int)

# 3. Soil Too Wet Flag
merged["soil_too_wet_flag"] = (merged["soil_moisture_0_to_1cm"] > 0.35).astype(int)

# 4. Poor Air Quality Flag
merged["poor_air_quality_flag"] = (merged["us_aqi"] > 100).astype(int)

# 5. High Pollen Flag
merged["high_pollen_flag"] = (
    merged[["grass_pollen", "ragweed_pollen", "birch_pollen",
            "alder_pollen", "mugwort_pollen", "olive_pollen"]]
    .fillna(0)
    .mean(axis=1) > 150
).astype(int)

# 6. Heat Stress Flag
merged["heat_stress_flag"] = (merged["temperature_2m"] >= 85).astype(int)

# 7. Respiratory Risk Flag
merged["respiratory_risk_flag"] = (merged["allergy_risk"] >= 60).astype(int)


In [ ]:
merged["best_overall_day_flag"] = (
    (merged["planting_readiness"] >= 65) &
    (merged["allergy_risk"] <= 40) &
    (merged["rain_expected_flag"] == 0) &
    (merged["high_wind_flag"] == 0) &
    (merged["poor_air_quality_flag"] == 0)
).astype(int)


In [ ]:
merged.to_csv("merged_open_meteo_final.csv", index=False)


In [ ]:
merged.head()

,date,temperature_2m,relative_humidity_2m,precipitation_probability,precipitation,wind_speed_10m,soil_temperature_0cm,soil_moisture_0_to_1cm,pm10,pm2_5,...,allergy_risk,best_day,high_wind_flag,rain_expected_flag,soil_too_wet_flag,poor_air_quality_flag,high_pollen_flag,heat_stress_flag,respiratory_risk_flag,best_overall_day_flag
0,2026-05-23 00:00:00,68.0666,93.0,82.0,0.141732,5.188683,66.837204,0.370,2.0,1.9,...,10.785714,0,0,1,1,0,0,0,0,0
1,2026-05-23 01:00:00,68.0666,96.0,71.0,0.125984,4.529580,66.477196,0.364,1.6,1.6,...,11.428571,0,0,1,1,0,0,0,0,0
2,2026-05-23 02:00:00,67.7966,96.0,42.0,0.000000,3.325540,66.747200,0.360,1.6,1.5,...,12.000000,0,0,0,1,0,0,0,0,0
3,2026-05-23 03:00:00,68.9666,95.0,32.0,0.000000,3.257121,66.837204,0.361,1.6,1.6,...,12.500000,0,0,0,1,0,0,0,0,0
4,2026-05-23 04:00:00,68.8766,93.0,36.0,0.015748,5.592500,67.287200,0.358,1.8,1.8,...,12.857143,0,0,0,1,0,0,0,0,0
